# 🚀 Stable Diffusion WebUI Forge Neo + SD-Hub (All-in-One Colab)
Mendukung model generasi baru (**Anima 2B**, **MOSS-Image**, **Z-Image-Turbo AIO**, **FLUX.2-Klein 4B**, dan **SDXL/NoobAI**) dengan tampilan klasik **AUTOMATIC1111**, manajemen memori anti-crash untuk GPU Colab T4, serta **SD-Hub** & **Direct Civitai Downloader** dengan **Civitai API Key otomatis**.

**Fitur Smart Check:** Otomatis melewati (*skip*) proses instalasi sistem, clone repo/ekstensi, pembuatan `venv`, dan download model yang sudah ada agar tidak pernah double download saat cell dijalankan ulang.

In [ ]:
# @title ⚙️ 1. Konfigurasi, Download Model & Jalankan Forge Neo + SD-Hub
# @markdown Pilih model yang ingin di-download (otomatis di-skip jika file sudah pernah di-download):

mount_google_drive = False # @param {type:"boolean"}

# @markdown ---
# @markdown ### 🔑 Konfigurasi Civitai API & Download via Link (Otomatis Injeksi Token)
# @markdown API Key ini otomatis dimasukkan ke **SD-Hub** di dalam WebUI dan dipakai untuk mendownload link Civitai di bawah (pisahkan dengan koma jika lebih dari 1 link):
civitai_api_key = "916ff2c247a9be5f8f3b0d3974af7863" # @param {type:"string"}
checkpoint_urls = "" # @param {type:"string"}
lora_urls = "" # @param {type:"string"}
vae_or_te_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ### 📦 Pilihan Model 1-File (Tanpa perlu atur VAE / Text Encoder)
download_Z_Image_Turbo_AIO = True # @param {type:"boolean"}
download_Anima_Pencil_XL = False # @param {type:"boolean"}

# @markdown ---
# @markdown ### 🧩 Pilihan Model Split (Otomatis download Checkpoint + Qwen Text Encoder + VAE)
download_Anima_2B = False # @param {type:"boolean"}
download_MOSS_Image_INT8 = False # @param {type:"boolean"}
download_Flux2_Klein_4B = False # @param {type:"boolean"}

# @markdown ---
# @markdown ### 🚀 Pengaturan WebUI
use_gradio_share = True # @param {type:"boolean"}
low_ram_protection = True # @param {type:"boolean"}
extra_args = "" # @param {type:"string"}

import os
import json
import re
import shutil
import urllib.request
from urllib.parse import urlparse, unquote, urlencode, parse_qsl, urlunparse

if mount_google_drive:
    if not os.path.ismount('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    else:
        print("✅ [Skip] Google Drive sudah ter-mount.")

# 1. Cek Dependensi Sistem & uv
print("🔍 [1/5] Mengecek dependensi sistem...")
if not (shutil.which("aria2c") and shutil.which("python3.11") and shutil.which("ffmpeg")):
    print("⏳ Menginstal aria2, python3.11, dan ffmpeg...")
    !apt-get update -qq
    !apt-get install -y aria2 git python3.11 python3.11-venv ffmpeg > /dev/null
else:
    print("✅ [Skip] Paket sistem (aria2, python3.11, ffmpeg) sudah terinstal.")

if not shutil.which("uv"):
    print("⏳ Menginstal uv...")
    !pip install -q uv
else:
    print("✅ [Skip] uv sudah terinstal.")

# 2. Cek Clone Repo, Ekstensi (MOSS & SD-Hub) & Virtual Environment
print("🔍 [2/5] Mengecek repository WebUI Forge Neo & Ekstensi (MOSS + SD-Hub)...")
WEBUI_DIR = "/content/forge-neo"

def clone_if_missing(repo_url, target_dir, branch=None):
    if os.path.exists(os.path.join(target_dir, ".git")):
        print(f"✅ [Skip] Repo sudah ada: {target_dir}")
    else:
        print(f"⏳ Meng-clone {repo_url} ke {target_dir}...")
        branch_arg = f"-b {branch}" if branch else ""
        !git clone {branch_arg} {repo_url} "{target_dir}"

clone_if_missing("https://github.com/Haoming02/sd-webui-forge-classic.git", WEBUI_DIR, branch="neo")
clone_if_missing("https://github.com/eduardoabreu81/moss-image-forge-neo.git", f"{WEBUI_DIR}/extensions/moss-image-forge-neo")
clone_if_missing("https://github.com/gutris1/sd-hub.git", f"{WEBUI_DIR}/extensions/sd-hub")

# 3. Otomatisasi Injeksi Civitai API Key ke SD-Hub
print("🔍 [3/5] Mengonfigurasi Civitai API Key otomatis untuk SD-Hub...")
SD_HUB_DIR = f"{WEBUI_DIR}/extensions/sd-hub"
sd_hub_cfg_files = [
    f"{SD_HUB_DIR}/.sd-hub-config.json",
    f"{WEBUI_DIR}/sd-hub-token.json"
]
sd_hub_payload = {
    "Token": {
        "civitai-api-key": civitai_api_key.strip(),
        "huggingface-read": "",
        "huggingface-write": ""
    },
    "civitai-api-key": civitai_api_key.strip()
}
for cfg_path in sd_hub_cfg_files:
    try:
        data = {}
        if os.path.exists(cfg_path):
            with open(cfg_path, "r") as f:
                data = json.load(f)
        data.setdefault("Token", {})["civitai-api-key"] = civitai_api_key.strip()
        data["civitai-api-key"] = civitai_api_key.strip()
        with open(cfg_path, "w") as f:
            json.dump(data, f, indent=4)
    except Exception as e:
        pass
print("✅ Civitai API Key berhasil dipasang otomatis ke SD-Hub!")

%cd {WEBUI_DIR}
if not os.path.exists(f"{WEBUI_DIR}/venv/bin/python"):
    print("⏳ Membuat virtual environment Python 3.11...")
    !uv venv venv --python 3.11 --seed
else:
    print("✅ [Skip] Virtual environment (venv) sudah ada.")

# 4. Cek & Download Model (Anti-Double Download + Support Link Civitai Otomatis)
print("🔍 [4/5] Mengecek & mengunduh model yang dipilih...")
CKPT_DIR = f"{WEBUI_DIR}/models/Stable-diffusion"
LORA_DIR = f"{WEBUI_DIR}/models/Lora"
TE_DIR = f"{WEBUI_DIR}/models/text_encoder"
VAE_DIR = f"{WEBUI_DIR}/models/VAE"
for d in [CKPT_DIR, LORA_DIR, TE_DIR, VAE_DIR]:
    os.makedirs(d, exist_ok=True)

def resolve_civitai_url(url, api_key):
    """Ubah link halaman Civitai (civitai.com/models/xxx) atau API download menjadi direct link dengan token + ambil nama file asli."""
    url = url.strip()
    filename = None
    if "civitai.com" in url:
        # Jika user paste link halaman model: https://civitai.com/models/12345?modelVersionId=67890
        m_ver = re.search(r"modelVersionId=(\d+)", url)
        m_mod = re.search(r"civitai\.com/models/(\d+)", url)
        try:
            if m_ver:
                ver_id = m_ver.group(1)
                req = urllib.request.Request(f"https://civitai.com/api/v1/model-versions/{ver_id}", headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(req, timeout=15) as resp:
                    info = json.loads(resp.read().decode())
                    filename = info["files"][0]["name"]
                    url = info["files"][0]["downloadUrl"]
            elif m_mod and "/api/download/" not in url:
                mod_id = m_mod.group(1)
                req = urllib.request.Request(f"https://civitai.com/api/v1/models/{mod_id}", headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(req, timeout=15) as resp:
                    info = json.loads(resp.read().decode())
                    ver_info = info["modelVersions"][0]
                    filename = ver_info["files"][0]["name"]
                    url = ver_info["files"][0]["downloadUrl"]
        except Exception as e:
            print(f"⚠️ Info Civitai API gagal diambil ({e}), lanjut menggunakan direct link...")

        # Tempelkan ?token=API_KEY jika belum ada
        if api_key:
            parsed = urlparse(url)
            qs = dict(parse_qsl(parsed.query))
            if "token" not in qs:
                qs["token"] = api_key
            url = urlunparse(parsed._replace(query=urlencode(qs)))
    return url, filename

def smart_dl(url, folder, filename=None):
    url, resolved_name = resolve_civitai_url(url, civitai_api_key.strip())
    if not filename:
        filename = resolved_name
    if not filename and "civitai.com" not in url:
        filename = os.path.basename(unquote(urlparse(url).path))

    if filename:
        target_path = os.path.join(folder, filename)
        aria2_ctrl = target_path + ".aria2"
        if os.path.exists(target_path) and not os.path.exists(aria2_ctrl) and os.path.getsize(target_path) > 1024 * 1024:
            size_gb = os.path.getsize(target_path) / (1024**3)
            print(f"✅ [Skip] {filename} sudah ada ({size_gb:.2f} GB).")
            return
        if os.path.exists(aria2_ctrl):
            print(f"🔄 Melanjutkan download yang sempat terputus: {filename}...")
        else:
            print(f"⬇️ Mengunduh {filename}...")
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M --allow-overwrite=false --auto-file-renaming=false "{url}" -d "{folder}" -o "{filename}"
    else:
        print(f"⬇️ Mengunduh dari link: {url.split('?')[0]}...")
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M --content-disposition=true --allow-overwrite=false --auto-file-renaming=false "{url}" -d "{folder}"

# 1. Z-Image-Turbo AIO (1 File)
if download_Z_Image_Turbo_AIO:
    smart_dl("https://huggingface.co/SeeSee21/Z-Image-Turbo-AIO/resolve/main/z-image-turbo-fp8-aio.safetensors", CKPT_DIR, "z-image-turbo-fp8-aio.safetensors")

# 2. Anima Pencil XL (1 File SDXL Anime)
if download_Anima_Pencil_XL:
    smart_dl("https://huggingface.co/bluepen5805/anima_pencil-XL/resolve/main/anima_pencil-XL_v500.safetensors", CKPT_DIR, "anima_pencil-XL_v500.safetensors")

# 3. Anima 2B Base (Split)
if download_Anima_2B:
    smart_dl("https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-base-v1.0.safetensors", CKPT_DIR, "anima-base-v1.0.safetensors")
    smart_dl("https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors", TE_DIR, "qwen_3_06b_base.safetensors")
    smart_dl("https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors", VAE_DIR, "qwen_image_vae.safetensors")

# 4. MOSS-Image INT8 (Split)
if download_MOSS_Image_INT8:
    smart_dl("https://huggingface.co/Comfy-Org/MOSS-Image/resolve/main/diffusion_models/moss_image_sft_int8_convrot.safetensors", CKPT_DIR, "moss_image_sft_int8_convrot.safetensors")
    smart_dl("https://huggingface.co/Comfy-Org/MOSS-Image/resolve/main/text_encoders/qwen3.5_9b_int8_convrot.safetensors", TE_DIR, "qwen3.5_9b_int8_convrot.safetensors")
    smart_dl("https://huggingface.co/Comfy-Org/MOSS-Image/resolve/main/vae/flux2-vae.safetensors", VAE_DIR, "flux2-vae.safetensors")

# 5. FLUX.2-Klein 4B (Split)
if download_Flux2_Klein_4B:
    smart_dl("https://huggingface.co/Comfy-Org/Flux-2-Klein-4B/resolve/main/split_files/diffusion_models/flux-2-klein-4b.safetensors", CKPT_DIR, "flux-2-klein-4b.safetensors")
    smart_dl("https://huggingface.co/Comfy-Org/Flux-2-Klein-4B/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors", TE_DIR, "qwen_3_4b.safetensors")
    smart_dl("https://huggingface.co/Comfy-Org/MOSS-Image/resolve/main/vae/flux2-vae.safetensors", VAE_DIR, "flux2-vae.safetensors")

# 6. Custom Links (Civitai / HuggingFace) - Otomatis pakai Civitai API Key
for u in [x.strip() for x in checkpoint_urls.split(",") if x.strip()]:
    smart_dl(u, CKPT_DIR)
for u in [x.strip() for x in lora_urls.split(",") if x.strip()]:
    smart_dl(u, LORA_DIR)
for u in [x.strip() for x in vae_or_te_urls.split(",") if x.strip()]:
    smart_dl(u, VAE_DIR)

print("🚀 [5/5] Menjalankan WebUI Forge Neo...")
args = ["--uv", "--enable-insecure-extension-access"]
if use_gradio_share:
    args.append("--share")
if low_ram_protection:
    args.append("--always-offload-from-vram")
if extra_args.strip():
    args.append(extra_args.strip())

cmd_args = " ".join(args)
with open("webui-user.sh", "w") as f:
    f.write(f'export COMMANDLINE_ARGS="{cmd_args}"\n')

!bash webui.sh -f